%% Install packages<br>
%pip install folium wget <br>
%% Import libraries

In [4]:
import folium, wget, pandas as pd
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon
from IPython.display import display
import plotly.express as px
# %% Download and read the `spacex_launch_geo.csv`
spacex_csv_file = wget.download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df=pd.read_csv(spacex_csv_file)
display(spacex_df.head())

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


%

Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`

In [5]:
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
display(launch_sites_df)

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


%% Start location is NASA Johnson Space Center

In [6]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name

In [7]:
circle = folium.Circle(nasa_coordinate, radius=1000, 
            color="#f3721b", fill=True).add_child(
            folium.Popup('NASA Johnson Space Center'))

Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name

In [14]:
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:%s;"><b>%s</b></div>' % ("#002ad3", 'NASA JSC'),
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

launch_sites_df['Launch Site']

0     CCAFS LC-40
1    CCAFS SLC-40
2      KSC LC-39A
3     VAFB SLC-4E
Name: Launch Site, dtype: str

%% Add a circle in the map for each launch site 

In [12]:
for site, lat, log in launch_sites_df.values:
    circle = folium.Circle([lat, log], radius=100, color="#9DF721",
              fill=True).add_child(folium.Popup(site))
    marker = folium.map.Marker([lat, log],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:%s;"><b>%s</b></div>' % ("#002ad3", site),
        )
    )
    site_map.add_child(circle)
    site_map.add_child(marker)

display(site_map)<br>
%% Mark the successed/failed launches for each site on the map <br>
Considering many launch records will have the same coordinates

In [11]:
marker_cluster = MarkerCluster()
spacex_df['marker_color'] = spacex_df['class'].map(lambda x: 
"#0A9103" if x==1 else "#C20000")
display(spacex_df[['marker_color','class']])
 
# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

,marker_color,class
0,#C20000,0
1,#C20000,0
2,#C20000,0
3,#C20000,0
4,#C20000,0
5,#C20000,0
6,#C20000,0
7,#C20000,0
8,#C20000,0
9,#C20000,0


for each row in spacex_df data frame create a Marker object with its coordinate<br>
and customize the Marker's icon property to indicate if this launch was successed or failed, <br>
e.g., icon=folium.Icon(color='white', icon_color=row['marker_color']

In [15]:
for index, record in spacex_df.iterrows():
    marker = folium.map.Marker([record.Lat, record.Long],
    icon=folium.Icon(
        color='white',
        icon_color = record.marker_color,
        icon='check' if record.marker_color == '#0A9103' else 'asterisk',
        )
    )
    marker_cluster.add_child(marker)

site_map

%% Calculate the distances between launch site to its proximities<br>
Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map

In [16]:
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

In [17]:
site_map.add_child(mouse_position)
# site_map

%%

In [ ]:
from math import sin, cos, sqrt, atan2, radians

In [18]:
def calculate_distance(lat_lon1, lat_lon2):
    lat1, lon1 = lat_lon1
    lat2, lon2 = lat_lon2
    # approximate radius of earth in km
    R = 6373.0
    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    distance = R * c
    return distance

%% Create and add a folium.Marker on your selected closest coastline point on the map<br>
Display the distance between coastline point and launch site using the icon property 

In [19]:
shore_coordinates = (28.56367 , -80.57163)
launch_coordinates = [launch_sites_df.loc[0,'Lat'],
                      launch_sites_df.loc[0,'Long']]

In [20]:
distance = calculate_distance(launch_coordinates, shore_coordinates)
distance_marker = folium.Marker(
   shore_coordinates,
   icon=DivIcon(
       icon_size=(20,20),
       icon_anchor=(0,0),
       html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>'
         % "{:10.2f} KM".format(distance),
       )
   )

NameError: name 'radians' is not defined

In [21]:
lines = folium.PolyLine(locations = [shore_coordinates, launch_coordinates],
                         weight=1)
site_map.add_child(lines)
# %%